Donor churn ML feature engineering notebook using Snowflake Feature Store
*Co-authored with CoCo*

# Donor Lapse/Churn Intelligence — (1 of 3) Feature Store, Datasets & Cortex ML Functions

This is **notebook 1 of 3** in the donor lapse/churn ML lab for a generic **nonprofit fundraising CRM**.

| Notebook | Builds |
|---|---|
| **01 — Features** *(this one)* | Feature Store (managed, point-in-time feature views), a versioned Dataset, and no-code Cortex ML Functions |
| 02 — Model | Snowpark ML training + Experiment Tracking + HPO, ML Jobs, Model Registry, Explainability |
| 03 — Serve & Agent | Batch/real-time serving, ML Observability, tool functions, and a Cortex Agent |

**Prerequisite:** run `lab/setup.sql` first. Requires `snowflake-ml-python >= 1.26`.

> The Feature Store here is the **real source of truth** for model features: managed feature views compute RFM / engagement / wealth signals from the raw `DONATIONS`, `ENGAGEMENTS`, and `DONORS` tables, versioned by a snapshot calendar so the *same* definitions serve training (as-of 12 months ago) and inference (as-of today) with no train/serve skew.

---
## Section 1: Connect & Explore

In [1]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

try:
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.getOrCreate()  # local run: uses your default connections.toml connection
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE DONOR_CHURN_ML_DEMO").collect()
session.sql("USE SCHEMA RAW").collect()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_WH").collect()
print("Connected to DONOR_CHURN_ML_DEMO")

import snowflake.ml.version
print("snowflake-ml-python:", snowflake.ml.version.VERSION)

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...


Connected to DONOR_CHURN_ML_DEMO
snowflake-ml-python: 1.46.0


In [2]:
# Quick label sanity check: lapsed vs retained donors should differ on RFM signals.
session.sql("""
    SELECT is_lapsed,
           COUNT(*)                         AS donors,
           ROUND(AVG(recency_days))         AS avg_recency_days,
           ROUND(AVG(frequency_last_12m),2) AS avg_gifts_12m,
           ROUND(AVG(engagement_last_6m),2) AS avg_engagement_6m,
           ROUND(AVG(monetary_total))       AS avg_lifetime_giving
    FROM DONOR_TRAINING_BASE
    GROUP BY is_lapsed
    ORDER BY is_lapsed
""").show()

---------------------------------------------------------------------------------------------------------------
|"IS_LAPSED"  |"DONORS"  |"AVG_RECENCY_DAYS"  |"AVG_GIFTS_12M"  |"AVG_ENGAGEMENT_6M"  |"AVG_LIFETIME_GIVING"  |
---------------------------------------------------------------------------------------------------------------
|0            |38011     |29                  |6.38             |5.71                 |42771                  |
|1            |11989     |404                 |1.88             |3.37                 |11235                  |
---------------------------------------------------------------------------------------------------------------



---
## Section 2: Feature Store (managed, point-in-time feature views)

We register a `DONOR` **Entity** and three **managed Feature Views** whose values are computed **from the raw tables** (`DONATIONS`, `ENGAGEMENTS`, `DONORS`) — not copied from a precomputed table. A small static **snapshot calendar** gives every feature a real timestamp, so point-in-time retrieval returns the correct as-of values for both training and serving.

| Feature View | Grain | Source | Features |
|---|---|---|---|
| `DONOR_RFM_FV` | donor × as-of | `DONATIONS` | tenure, frequency (lifetime / 12m), monetary total, avg gift, recency |
| `DONOR_ENGAGEMENT_FV` | donor × as-of | `ENGAGEMENTS` | engagement count, engagement (6m) |
| `DONOR_STATIC_FV` | donor | `DONORS` | wealth signal score |

Managed feature views are backed by Dynamic Tables, so their definitions must be **deterministic** — that's why the as-of dates come from a static `SNAPSHOT_CALENDAR` table (never `CURRENT_DATE()` inside the view).

In [3]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="DONOR_CHURN_ML_DEMO",
    name="FEATURES",
    default_warehouse="DONOR_CHURN_ML_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

donor = Entity(name="DONOR", join_keys=["DONOR_ID"], desc="A donor of the nonprofit fundraising CRM")
fs.register_entity(donor)
print("Entities:", [e["NAME"] for e in fs.list_entities().collect()])

/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:463: UserWarning: Entity DONOR already exists. Skip registration.
  return f(self, *args, **kargs)


Entities: ['DONOR']


In [4]:
# Static snapshot calendar: monthly as-of dates for the trailing 18 months.
# Built ONCE as a plain table (CURRENT_DATE is fine here — it is NOT inside a Dynamic Table),
# so the managed feature views below can reference it deterministically.
session.sql("USE SCHEMA FEATURES").collect()
session.sql("""
    CREATE OR REPLACE TABLE DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR AS
    SELECT DATEADD('month', -seq4(), DATE_TRUNC('month', CURRENT_DATE()))::TIMESTAMP_NTZ AS as_of_ts
    FROM TABLE(GENERATOR(ROWCOUNT => 18))
""").collect()
session.sql("SELECT MIN(as_of_ts) earliest, MAX(as_of_ts) latest, COUNT(*) snapshots FROM DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR").show()

-----------------------------------------------------------
|"EARLIEST"           |"LATEST"             |"SNAPSHOTS"  |
-----------------------------------------------------------
|2025-02-01 00:00:00  |2026-07-01 00:00:00  |18           |
-----------------------------------------------------------



In [5]:
# DONOR_RFM_FV — point-in-time RFM from DONATIONS. One row per (donor, as-of snapshot),
# aggregating only gifts on or before that snapshot. recency/tenure are relative to the snapshot.
rfm_df = session.sql("""
    SELECT c.as_of_ts,
           d.donor_id,
           DATEDIFF('day', d.acquisition_date, c.as_of_ts)                       AS tenure_days,
           COUNT(dn.gift_id)                                                     AS frequency_lifetime,
           COUNT(CASE WHEN dn.gift_date >= DATEADD('month', -12, c.as_of_ts) THEN 1 END) AS frequency_last_12m,
           COALESCE(SUM(dn.gift_amount), 0)                                      AS monetary_total,
           COALESCE(AVG(dn.gift_amount), 0)                                      AS avg_gift_amount,
           COALESCE(DATEDIFF('day', MAX(dn.gift_date), c.as_of_ts), 1500)        AS recency_days
    FROM DONOR_CHURN_ML_DEMO.RAW.DONORS d
    CROSS JOIN DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR c
    LEFT JOIN DONOR_CHURN_ML_DEMO.RAW.DONATIONS dn
           ON dn.donor_id = d.donor_id AND dn.gift_date <= c.as_of_ts
    WHERE d.acquisition_date <= c.as_of_ts
    GROUP BY c.as_of_ts, d.donor_id, d.acquisition_date
""")

donor_rfm_fv = FeatureView(
    name="DONOR_RFM_FV", entities=[donor], feature_df=rfm_df,
    timestamp_col="AS_OF_TS", refresh_freq="1 day",
    desc="Point-in-time RFM (recency, frequency, monetary, tenure) per donor from DONATIONS",
)
donor_rfm_fv = fs.register_feature_view(feature_view=donor_rfm_fv, version="v1", overwrite=True)
print("Registered DONOR_RFM_FV/v1")

/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:5443: UserWarning: Your pipeline won't be incrementally refreshed due to: "Change tracking is not supported on queries containing outer joins with non-equality predicates.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered DONOR_RFM_FV/v1


In [6]:
# DONOR_ENGAGEMENT_FV — point-in-time engagement counts from ENGAGEMENTS.
eng_df = session.sql("""
    SELECT c.as_of_ts,
           d.donor_id,
           COUNT(en.engagement_id)                                              AS engagement_count,
           COUNT(CASE WHEN en.event_date >= DATEADD('month', -6, c.as_of_ts) THEN 1 END) AS engagement_last_6m
    FROM DONOR_CHURN_ML_DEMO.RAW.DONORS d
    CROSS JOIN DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR c
    LEFT JOIN DONOR_CHURN_ML_DEMO.RAW.ENGAGEMENTS en
           ON en.donor_id = d.donor_id AND en.event_date <= c.as_of_ts
    WHERE d.acquisition_date <= c.as_of_ts
    GROUP BY c.as_of_ts, d.donor_id
""")

donor_eng_fv = FeatureView(
    name="DONOR_ENGAGEMENT_FV", entities=[donor], feature_df=eng_df,
    timestamp_col="AS_OF_TS", refresh_freq="1 day",
    desc="Point-in-time engagement counts per donor from ENGAGEMENTS",
)
donor_eng_fv = fs.register_feature_view(feature_view=donor_eng_fv, version="v1", overwrite=True)
print("Registered DONOR_ENGAGEMENT_FV/v1")

/Users/snuggehalli/Documents/Internal_Demos/field-demo-enablement/donor-churn-ml/.venv-ml/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:5443: UserWarning: Your pipeline won't be incrementally refreshed due to: "Change tracking is not supported on queries containing outer joins with non-equality predicates.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Registered DONOR_ENGAGEMENT_FV/v1


In [7]:
# DONOR_STATIC_FV — slowly-changing donor attribute (wealth signal). No timestamp needed.
static_df = session.sql("""
    SELECT donor_id, wealth_signal_score
    FROM DONOR_CHURN_ML_DEMO.RAW.DONORS
""")
donor_static_fv = FeatureView(
    name="DONOR_STATIC_FV", entities=[donor], feature_df=static_df,
    refresh_freq="1 day", desc="Static donor attributes (wealth signal score)",
)
donor_static_fv = fs.register_feature_view(feature_view=donor_static_fv, version="v1", overwrite=True)
print("Registered DONOR_STATIC_FV/v1")
fs.list_feature_views().select("NAME", "VERSION").show()

Registered DONOR_STATIC_FV/v1


-----------------------------------
|"NAME"               |"VERSION"  |
-----------------------------------
|DONOR_ENGAGEMENT_FV  |v1         |
|DONOR_RFM_FV         |v1         |
|DONOR_STATIC_FV      |v1         |
|DONOR_RFM_FEATURES   |v1         |
-----------------------------------



In [8]:
# Prove point-in-time correctness: the SAME feature view yields DIFFERENT values
# at the training as-of (12 months ago) vs the most recent snapshot.
session.sql("""
    SELECT as_of_ts,
           COUNT(*)                        AS donors,
           ROUND(AVG(frequency_last_12m),2) AS avg_gifts_12m,
           ROUND(AVG(recency_days))         AS avg_recency_days,
           ROUND(AVG(monetary_total))       AS avg_lifetime_giving
    FROM DONOR_CHURN_ML_DEMO.FEATURES."DONOR_RFM_FV$v1"
    WHERE as_of_ts IN (
        (SELECT MIN(as_of_ts) FROM DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR),
        (SELECT MAX(as_of_ts) FROM DONOR_CHURN_ML_DEMO.FEATURES.SNAPSHOT_CALENDAR))
    GROUP BY as_of_ts ORDER BY as_of_ts
""").show()

-------------------------------------------------------------------------------------------------
|"AS_OF_TS"           |"DONORS"  |"AVG_GIFTS_12M"  |"AVG_RECENCY_DAYS"  |"AVG_LIFETIME_GIVING"  |
-------------------------------------------------------------------------------------------------
|2025-02-01 00:00:00  |48456     |5.55             |92                  |33019                  |
|2026-07-01 00:00:00  |50000     |4.56             |224                 |41561                  |
-------------------------------------------------------------------------------------------------



---
## Section 3: Datasets

Materialize an **immutable, versioned training set** by joining a **spine** (donor + as-of date + label) to the feature views with a point-in-time join. The label `is_lapsed` = *no gift in the 12 months after the as-of date*, computed at the **same** as-of instant as the features — so there is no leakage and no train/label skew.

In [9]:
# Spine: as-of = the training snapshot (~12 months ago); label from the forward 12-month window.
spine_df = session.sql("""
    WITH as_of AS (
        SELECT DATEADD('month', -12, DATE_TRUNC('month', CURRENT_DATE()))::TIMESTAMP_NTZ AS as_of_ts
    ),
    fwd AS (
        SELECT d.donor_id,
               COUNT(dn.gift_id) AS gifts_forward_12m
        FROM DONOR_CHURN_ML_DEMO.RAW.DONORS d
        CROSS JOIN as_of a
        LEFT JOIN DONOR_CHURN_ML_DEMO.RAW.DONATIONS dn
               ON dn.donor_id = d.donor_id
              AND dn.gift_date >  a.as_of_ts
              AND dn.gift_date <= DATEADD('month', 12, a.as_of_ts)
        WHERE d.acquisition_date <= a.as_of_ts
        GROUP BY d.donor_id
    )
    SELECT f.donor_id, a.as_of_ts,
           IFF(COALESCE(f.gifts_forward_12m, 0) = 0, 1, 0) AS is_lapsed
    FROM fwd f CROSS JOIN as_of a
""")
print("Spine rows:", spine_df.count())

Spine rows: 50000


In [10]:
from snowflake.ml.dataset import Dataset

try:
    training_dataset = fs.generate_dataset(
        name="DONOR_CHURN_ML_DEMO.FEATURES.LAPSE_TRAINING_SET",
        version="v1",
        spine_df=spine_df,
        features=[donor_rfm_fv, donor_eng_fv, donor_static_fv],
        spine_timestamp_col="AS_OF_TS",
        spine_label_cols=["IS_LAPSED"],
        desc="Point-in-time donor lapse training set (features retrieved through the Feature Store)",
    )
    print("Materialized dataset LAPSE_TRAINING_SET/v1")
except RuntimeError as e:
    if "already exists" in str(e):
        training_dataset = Dataset(session, "DONOR_CHURN_ML_DEMO", "FEATURES", "LAPSE_TRAINING_SET", "v1")
        print("Dataset LAPSE_TRAINING_SET/v1 already exists — loaded existing version.")
    else:
        raise

train_sp_df = training_dataset.read.to_snowpark_dataframe()
train_sp_df.select("DONOR_ID", "IS_LAPSED", "RECENCY_DAYS", "FREQUENCY_LAST_12M", "WEALTH_SIGNAL_SCORE").limit(5).show()

Materialized dataset LAPSE_TRAINING_SET/v1


--------------------------------------------------------------------------------------------
|"DONOR_ID"  |"IS_LAPSED"  |"RECENCY_DAYS"  |"FREQUENCY_LAST_12M"  |"WEALTH_SIGNAL_SCORE"  |
--------------------------------------------------------------------------------------------
|7           |0            |35              |6                     |0.3160000145435333     |
|26          |0            |37              |8                     |0.5360000133514404     |
|39          |0            |22              |7                     |0.13699999451637268    |
|44          |0            |34              |6                     |0.6380000114440918     |
|50          |1            |383             |0                     |0.36800000071525574    |
--------------------------------------------------------------------------------------------



---
## Section 4: Cortex ML Functions (no-code bridge)

`setup.sql` created three built-in **Cortex ML Function** objects. Here we exercise them: **Forecast**, **Anomaly Detection**, and **Classification** (a no-code baseline lapse model + feature importance) — a fast, SQL-only bridge before the custom Snowpark ML model in notebook 02.

In [11]:
# 4a. FORECAST — next 13 weeks (~1 quarter) of donation volume.
session.sql("USE SCHEMA MODELS").collect()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_SOWH").collect()
session.sql("CALL DONATION_VOLUME_FORECAST!FORECAST(FORECASTING_PERIODS => 13)").show()

--------------------------------------------------------------------------------------------
|"SERIES"  |"TS"                 |"FORECAST"        |"LOWER_BOUND"      |"UPPER_BOUND"     |
--------------------------------------------------------------------------------------------
|NULL      |2026-07-13 00:00:00  |1425102.45679228  |386197.253962245   |2464007.61497946  |
|NULL      |2026-07-20 00:00:00  |1207048.61961128  |-232398.43844886   |2646495.6866      |
|NULL      |2026-07-27 00:00:00  |1216372.40743029  |-534261.230087442  |2967006.01816231  |
|NULL      |2026-08-03 00:00:00  |1261865.07024929  |-753254.696507003  |3276984.75664845  |
|NULL      |2026-08-10 00:00:00  |1194582.4830683   |-1055053.47744629  |3444218.42572575  |
|NULL      |2026-08-17 00:00:00  |1165587.6458873   |-1297243.94383585  |3628419.21775331  |
|NULL      |2026-08-24 00:00:00  |1174880.55870631  |-1484992.17776175  |3834753.21481722  |
|NULL      |2026-08-31 00:00:00  |1145885.72152531  |-1698252.53481898

In [12]:
# 4b. ANOMALY DETECTION — flag unusual weeks of giving drop-off / spikes.
session.sql("""
    SELECT * FROM TABLE(DONOR_CHURN_ML_DEMO.MODELS.DONATION_VOLUME_ANOMALY!DETECT_ANOMALIES(
        INPUT_DATA => SYSTEM$QUERY_REFERENCE('SELECT * FROM DONOR_CHURN_ML_DEMO.RAW.DONATION_VOLUME_TS WHERE WEEK_TS >= DATEADD(MONTH, -6, CURRENT_DATE())'),
        TIMESTAMP_COLNAME => 'WEEK_TS',
        TARGET_COLNAME => 'TOTAL_AMOUNT'))
    WHERE is_anomaly = TRUE
    ORDER BY ts
""").show()

--------------------------------------------------------------------------------------------------------------------------------------------------------
|"SERIES"  |"TS"                 |"Y"         |"FORECAST"        |"LOWER_BOUND"     |"UPPER_BOUND"     |"IS_ANOMALY"  |"PERCENTILE"     |"DISTANCE"    |
--------------------------------------------------------------------------------------------------------------------------------------------------------
|NULL      |2026-06-08 00:00:00  |3448774.4   |7683254.47683749  |4366730.75208457  |10999778.2015904  |True          |0.0005031234664  |-3.288774263  |
|NULL      |2026-06-15 00:00:00  |2835601.4   |7657382.83140495  |4340859.10665203  |10973906.5561579  |True          |9.022889726e-05  |-3.744910918  |
|NULL      |2026-06-22 00:00:00  |2550656.05  |7805595.67788227  |4489071.95312935  |11122119.4026352  |True          |2.238937045e-05  |-4.081329912  |
|NULL      |2026-06-29 00:00:00  |2410711.3   |7556920.85676204  |4240397.13200912

In [13]:
# 4c. CLASSIFICATION — no-code baseline lapse model: evaluation + feature importance.
session.sql("CALL LAPSE_BASELINE_MODEL!SHOW_EVALUATION_METRICS()").show()
session.sql("CALL LAPSE_BASELINE_MODEL!SHOW_FEATURE_IMPORTANCE()").show()
session.sql("USE WAREHOUSE DONOR_CHURN_ML_WH").collect()

-----------------------------------------------------------------------
|"DATASET_TYPE"  |"CLASS"  |"ERROR_METRIC"  |"METRIC_VALUE"  |"LOGS"  |
-----------------------------------------------------------------------
|EVAL            |0        |precision       |0.9944954128    |NULL    |
|EVAL            |0        |recall          |0.9756975698    |NULL    |
|EVAL            |0        |f1              |0.9850068151    |NULL    |
|EVAL            |0        |support         |1111.0          |NULL    |
|EVAL            |1        |precision       |0.9497206704    |NULL    |
|EVAL            |1        |recall          |0.988372093     |NULL    |
|EVAL            |1        |f1              |0.9686609687    |NULL    |
|EVAL            |1        |support         |516.0           |NULL    |
-----------------------------------------------------------------------



-----------------------------------------------------------------
|"RANK"  |"FEATURE"            |"SCORE"        |"FEATURE_TYPE"  |
-----------------------------------------------------------------
|1       |RECENCY_DAYS         |0.203          |user_provided   |
|2       |TENURE_DAYS          |0.134          |user_provided   |
|3       |ENGAGEMENT_COUNT     |0.1336666667   |user_provided   |
|4       |WEALTH_SIGNAL_SCORE  |0.1293333333   |user_provided   |
|5       |FREQUENCY_LIFETIME   |0.09633333333  |user_provided   |
|6       |ENGAGEMENT_LAST_6M   |0.081          |user_provided   |
|7       |MONETARY_TOTAL       |0.07566666667  |user_provided   |
|8       |AVG_GIFT_AMOUNT      |0.074          |user_provided   |
|9       |FREQUENCY_LAST_12M   |0.05066666667  |user_provided   |
|10      |CHANNEL              |0.017          |user_provided   |
-----------------------------------------------------------------



[Row(status='Statement executed successfully.')]

---
## Notebook 1 complete

You registered a `DONOR` entity, three **managed point-in-time feature views** computed from raw data, and a versioned **Dataset** — plus exercised the no-code Cortex ML Functions.

| Object | Name |
|---|---|
| Entity | `DONOR` |
| Feature Views | `DONOR_RFM_FV/v1`, `DONOR_ENGAGEMENT_FV/v1`, `DONOR_STATIC_FV/v1` |
| Dataset | `FEATURES.LAPSE_TRAINING_SET/v1` |

**Next:** open **`donor-churn-02-model.ipynb`** to train, track experiments, tune, and register the model.